# Estructuras de datos y almacenamiento

Elegir una estructura de datos no es una decisión de estilo: decide cuánta
RAM se usa, qué tan rápido corre el código, y en algunos casos si el
resultado es correcto o no. Este notebook mide 4 cosas concretas sobre
nuestros propios datos, no sobre un ejemplo genérico: cuánto pesa una lista
de Python frente a un arreglo de NumPy, cuánta memoria ahorran los tipos de
columna bien elegidos, cuánto pesa y cuánto tarda en leerse el mismo
dataset en CSV contra Parquet, y dos errores de pandas que no lanzan
ninguna excepción pero dan un resultado incorrecto.

No depende del catálogo de Kedro: todas las rutas son relativas desde
`notebooks/` hacia `data/`.

## Bloque 1 — Lista vs NumPy: memoria y velocidad

Una lista de Python es un arreglo de punteros: cada elemento es un objeto
`float` completo, disperso en memoria, con su propio encabezado de tipo y
su propio conteo de referencias. Un `ndarray` de NumPy es un bloque
contiguo de bytes del mismo tipo, sin envoltorio por elemento, y sus
operaciones corren en código C vectorizado en vez de un bucle interpretado
de Python. La diferencia no es solo de memoria: al vectorizar, NumPy evita
también el costo de interpretar cada iteración del bucle.

In [1]:
import sys, time
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/02_intermediate/detecciones_limpias.parquet")

# Memoria: lista vs numpy
velocidades_lista = df["speed_mps"].dropna().tolist()
velocidades_array = np.array(velocidades_lista, dtype="float64")

# Memoria real de la lista (punteros + objetos)
bytes_lista = sys.getsizeof(velocidades_lista) + sum(sys.getsizeof(x) for x in velocidades_lista)
bytes_array = velocidades_array.nbytes

print(f"Elementos: {len(velocidades_lista):,}")
print(f"Lista Python:  {bytes_lista/1024**2:.2f} MB")
print(f"Array NumPy:   {bytes_array/1024**2:.2f} MB")
print(f"Factor:        {bytes_lista/bytes_array:.1f}×")

Elementos: 38,879
Lista Python:  1.19 MB
Array NumPy:   0.30 MB
Factor:        4.0×


In [2]:
# Velocidad: for vs vectorizado
inicio = time.perf_counter()
resultado_loop = [v * 3.6 for v in velocidades_lista]
t_loop = time.perf_counter() - inicio

inicio = time.perf_counter()
resultado_vec = velocidades_array * 3.6
t_vec = time.perf_counter() - inicio

print(f"Loop Python:   {t_loop*1000:.2f} ms")
print(f"Vectorizado:   {t_vec*1000:.2f} ms")
print(f"Factor:        {t_loop/t_vec:.0f}× más rápido")

Loop Python:   0.96 ms
Vectorizado:   0.07 ms
Factor:        13× más rápido


In [3]:
factor_memoria = bytes_lista / bytes_array
assert factor_memoria >= 2, f"se esperaba al menos 2x de eficiencia en memoria, se obtuvo {factor_memoria:.1f}x"
print(f"OK: el array usa {factor_memoria:.1f}x menos memoria que la lista (>= 2x esperado).")

OK: el array usa 4.0x menos memoria que la lista (>= 2x esperado).


## Bloque 2 — Optimización de tipos de columna

Pandas infiere, por defecto, los tipos más generosos que puede: texto libre
queda como `object` (un puntero a un objeto Python por celda, igual que una
lista) y los números quedan en `float64` aunque no se necesite esa
precisión. Ajustar los tipos después de conocer los datos —`category` para
texto con pocos valores distintos, `float32` donde no hace falta doble
precisión— puede reducir la memoria a la mitad sin perder ninguna
información real.

In [4]:
memoria_antes = df.memory_usage(deep=True).sum() / 1024**2

df_opt = df.copy()
# object → category para columnas categóricas
for col in ["object_type", "weather", "time_of_day", "location"]:
    if col in df_opt.columns:
        df_opt[col] = df_opt[col].astype("category")

# float64 → float32 donde no se necesita precisión doble
for col in df_opt.select_dtypes("float64").columns:
    df_opt[col] = df_opt[col].astype("float32")

memoria_despues = df_opt.memory_usage(deep=True).sum() / 1024**2
ahorro = (1 - memoria_despues/memoria_antes) * 100

print(f"Memoria antes:  {memoria_antes:.2f} MB")
print(f"Memoria después: {memoria_despues:.2f} MB")
print(f"Ahorro:         {ahorro:.1f}%")

Memoria antes:  18.46 MB
Memoria después: 9.72 MB
Ahorro:         47.4%


In [5]:
assert ahorro > 30, f"se esperaba un ahorro mayor a 30%, se obtuvo {ahorro:.1f}%"
print(f"OK: el ahorro de memoria ({ahorro:.1f}%) supera el 30% esperado.")

OK: el ahorro de memoria (47.4%) supera el 30% esperado.


## Bloque 3 — Benchmark CSV vs Parquet

El CSV no guarda tipos: es texto plano, y al releerlo pandas tiene que
volver a adivinar qué tipo tiene cada columna. Parquet sí guarda tipos,
junto con los datos, en su propio metadato binario. Esa no es una
diferencia de detalle, es la razón concreta por la que el pipeline de Kedro
(`preprocesamiento`, `ingesta_waymo`) guarda todos sus datasets
intermedios y finales en Parquet, nunca en CSV.

Esta comparación se hace sobre `df_opt` (el dataset con los tipos ya
optimizados del Bloque 2), no sobre `df`. La razón: con los tipos por
defecto de pandas (`object` y `float64`), un CSV no pierde nada demostrable
porque texto y `float64` sobreviven un viaje de ida y vuelta por CSV sin
cambiar de tipo. La pérdida de tipo solo se vuelve visible, y relevante,
una vez que hay tipos más específicos que preservar (`category`,
`float32`), que es exactamente lo que este proyecto querría conservar en
sus datasets procesados.

In [6]:
import tempfile, os
from pathlib import Path

tmp = Path(tempfile.mkdtemp())

# Guardar en ambos formatos
df_opt.to_csv(tmp / "muestra.csv", index=False)
df_opt.to_parquet(tmp / "muestra.parquet", index=False)

peso_csv = os.path.getsize(tmp / "muestra.csv") / 1024**2
peso_parquet = os.path.getsize(tmp / "muestra.parquet") / 1024**2

# Tiempo de lectura
inicio = time.perf_counter()
pd.read_csv(tmp / "muestra.csv")
t_csv = time.perf_counter() - inicio

inicio = time.perf_counter()
pd.read_parquet(tmp / "muestra.parquet")
t_parquet = time.perf_counter() - inicio

print(f"Peso CSV:     {peso_csv:.2f} MB")
print(f"Peso Parquet: {peso_parquet:.2f} MB")
print(f"Ratio peso:   {peso_csv/peso_parquet:.1f}× más pesado el CSV")
print(f"Lectura CSV:  {t_csv*1000:.0f} ms")
print(f"Lectura Parquet: {t_parquet*1000:.0f} ms")

Peso CSV:     5.52 MB
Peso Parquet: 1.90 MB
Ratio peso:   2.9× más pesado el CSV
Lectura CSV:  95 ms
Lectura Parquet: 25 ms


In [7]:
tipos_originales = df_opt.dtypes
releido_parquet = pd.read_parquet(tmp / "muestra.parquet")  # parquet conserva
releido_csv2 = pd.read_csv(tmp / "muestra.csv")               # csv no conserva

perdidas = []
for col in df_opt.columns:
    if str(tipos_originales[col]) != str(releido_csv2[col].dtype):
        perdidas.append({
            "columna": col,
            "tipo_original": str(tipos_originales[col]),
            "tipo_tras_csv": str(releido_csv2[col].dtype),
        })

perdidas_df = pd.DataFrame(perdidas)
print(f"Columnas que cambian de tipo al pasar por CSV: {len(perdidas)}")
print(perdidas_df.to_string(index=False))

perdidas_parquet = sum(
    str(tipos_originales[col]) != str(releido_parquet[col].dtype) for col in df_opt.columns
)
print(f"\nColumnas que cambian de tipo al pasar por Parquet: {perdidas_parquet}")

Columnas que cambian de tipo al pasar por CSV: 15
         columna tipo_original tipo_tras_csv
timestamp_micros       float32       float64
     object_type      category        object
    box_center_x       float32       float64
    box_center_y       float32       float64
    box_center_z       float32       float64
      box_length       float32       float64
       box_width       float32       float64
      box_height       float32       float64
       speed_mps       float32       float64
num_lidar_points       float32       float64
         weather      category        object
     time_of_day      category        object
     distancia_m       float32       float64
    volumen_caja       float32       float64
 densidad_puntos       float32       float64

Columnas que cambian de tipo al pasar por Parquet: 0


In [8]:
assert len(perdidas) >= 1, "se esperaba que CSV perdiera al menos 1 tipo"
assert perdidas_parquet == 0, "se esperaba que Parquet conservara todos los tipos"
print(f"OK: CSV pierde el tipo de {len(perdidas)} columnas, Parquet conserva las {df_opt.shape[1]}.")

OK: CSV pierde el tipo de 15 columnas, Parquet conserva las 18.


## Bloque 4 — Las trampas de pandas

Dos errores que no dan una excepción: dan un resultado silenciosamente
incorrecto, que solo se nota si alguien revisa el resultado con cuidado.

In [9]:
muestra = df.sample(5, random_state=42)
# .iloc usa posición (0, 1, 2...) — siempre funciona
print("iloc[0]:", muestra.iloc[0]["object_type"])
# .loc usa el índice del DataFrame — si está desordenado, no es 0
print("loc[0]: puede dar KeyError o un resultado inesperado si el índice no empieza en 0")
try:
    print(muestra.loc[0]["object_type"])
except KeyError:
    print("KeyError: el índice no tiene 0 porque es una muestra aleatoria (esperado)")

iloc[0]: VEHICLE
loc[0]: puede dar KeyError o un resultado inesperado si el índice no empieza en 0
KeyError: el índice no tiene 0 porque es una muestra aleatoria (esperado)


Esto no es un caso límite raro: `df.sample()`, cualquier `.query()`, o
cualquier filtro booleano (`df[df["object_type"] == "CYCLIST"]`) deja el
índice original desordenado. Usar `.loc[0]` después de eso no da "la
primera fila", da "la fila cuyo índice es literalmente 0", si es que
existe. Si el 0 sí estuviera en el índice de la muestra por casualidad, no
habría ningún error, `.loc[0]` devolvería una fila, solo que no la que se
esperaba.

In [10]:
s1 = pd.Series([1, 2, 3], index=[0, 1, 2])
s2 = pd.Series([10, 20, 30], index=[1, 2, 3])
resultado = s1 + s2
print("Resultado de sumar series con índices distintos:")
print(resultado)
print("NaN en posición 0 y 3 — pandas alinea por índice, no por posición")

Resultado de sumar series con índices distintos:
0     NaN
1    12.0
2    23.0
3     NaN
dtype: float64
NaN en posición 0 y 3 — pandas alinea por índice, no por posición


`s1` y `s2` tienen 3 elementos cada una, y la suma "debería" dar 3
resultados si pandas alineara por posición. En cambio, pandas alinea por
**índice**: solo las etiquetas 1 y 2 existen en ambas series, así que esas
dos suman de verdad (1+10=11, 2+20=22), y las etiquetas 0 y 3, que solo
existen en una de las dos series, dan `NaN` en vez de un error. Sin mirar
el resultado con atención, es fácil no notar que 2 de los 4 valores del
resultado son `NaN`.

## Bloque 5 — Decisión documentada para el proyecto

In [11]:
decision = {
    "formato_datos_crudos": "CSV",
    "razon_datos_crudos": "formato de entrega del docente, compatible con cualquier herramienta",
    "formato_datos_procesados": "Parquet",
    "razon_datos_procesados": f"conserva tipos, {peso_csv/peso_parquet:.1f}× más compacto que CSV, {t_csv/t_parquet:.1f}× más rápido de leer",
    "formato_datos_waymo": "Parquet",
    "razon_waymo": "formato nativo del Waymo Open Dataset v2, ya viene como Parquet del bucket GCS",
    "estructura_manipulacion": "pandas DataFrame con tipos optimizados",
    "razon_estructura": f"ahorro de {ahorro:.1f}% de memoria con category y float32 vs object y float64",
}

for clave, valor in decision.items():
    print(f"{clave:35s}: {valor}")

formato_datos_crudos               : CSV
razon_datos_crudos                 : formato de entrega del docente, compatible con cualquier herramienta
formato_datos_procesados           : Parquet
razon_datos_procesados             : conserva tipos, 2.9× más compacto que CSV, 3.8× más rápido de leer
formato_datos_waymo                : Parquet
razon_waymo                        : formato nativo del Waymo Open Dataset v2, ya viene como Parquet del bucket GCS
estructura_manipulacion            : pandas DataFrame con tipos optimizados
razon_estructura                   : ahorro de 47.4% de memoria con category y float32 vs object y float64


In [12]:
assert decision["formato_datos_procesados"] == "Parquet"
print("OK: el formato de datos procesados documentado es Parquet.")

OK: el formato de datos procesados documentado es Parquet.


## Cierre — Tabla de decisiones de almacenamiento

| Medición | Cifra |
|---|---|
| Filas × columnas (dataset limpio) | 39.800 × 18 |
| Memoria al cargar (`deep=True`) | 18,46 MB |
| Memoria tras optimizar tipos | 9,72 MB |
| Ahorro conseguido | 47,4% |
| Peso en CSV (tipos optimizados) | 5,52 MB |
| Peso en Parquet (tipos optimizados) | 1,90 MB |
| Columnas que CSV pierde de tipo | 15 de 18 |

**Las 3 trampas:**

- `sys.getsizeof` sobre una lista miente: solo mide el arreglo de
  punteros (104 bytes fijos), no el peso real de los objetos `float` que
  apuntan. Sumando el tamaño de cada elemento (Bloque 1), la lista real
  pesa 4,0 veces más que el array de NumPy equivalente, no una cifra
  cercana a cero como sugeriría mirar solo `sys.getsizeof(lista)`.
- Asignar (u operar) series con índice distinto no da error, da `NaN`:
  demostrado en el Bloque 4, `s1 + s2` con índices `[0,1,2]` y `[1,2,3]`
  da `NaN` en las etiquetas 0 y 3.
- Guardar en CSV borra los tipos: 15 de 18 columnas (todas las `category`
  y todas las `float32`) cambian de tipo al pasar por un CSV y releerse;
  Parquet conserva las 18.

**Si el dataset creciera de 39.800 filas a 100 millones, ¿qué dejaría de
funcionar?** Pandas carga todo el DataFrame en memoria RAM de un solo
proceso: con 100 millones de filas y un ancho de columnas similar al
actual, la memoria requerida escalaría por un factor de más de 2.500x
sobre los 9,72 MB medidos tras optimizar tipos, muy por encima de lo que
una máquina de desarrollo típica puede mantener en memoria junto con el
resto del sistema. `df.copy()`, cualquier `.groupby()` sin optimizar, o
simplemente cargar dos versiones del dataset a la vez (crudo y limpio)
dejaría de ser viable mucho antes de llegar a esa cifra. La salida no es
"pandas pero con más RAM": es migrar el procesamiento a un motor
distribuido (Spark o su variante gestionada en Databricks), que reparte
los datos y el cómputo entre varias máquinas en vez de una sola, sobre el
mismo formato Parquet que este notebook ya usa: Parquet particiona bien y
Spark lo lee de forma nativa, así que la decisión de formato tomada acá
seguiría siendo válida incluso si cambiara el motor de cómputo. Esto
todavía no está documentado en el README como arquitectura futura; es una
extensión natural de la sección 6 (CRISP-DM, fase Deployment), no algo que
este proyecto necesite implementar hoy.